## LLM Fallback for Low-Confidence BERT Predictions

Uses Groq (Llama-3.1-70B) to re-classify entity pairs where BERT confidence is below threshold.
Only called for rare/ambiguous predicates — keeps API calls minimal.

In [ ]:
import os, json, time
from pathlib import Path
from collections import defaultdict
import torch
from groq import Groq

# ════════════════════════════════════════════
# CONFIGURE HERE
# ════════════════════════════════════════════
GROQ_API_KEY = "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"   # <-- inserisci la tua API key

# Logits cache del tuo miglior modello BERT
# Cambia questo path con il modello migliore dopo i confronti
BERT_LOGITS_CACHE = Path("models/bert_biomedbert_re_A0_fixed/dev_logits_cache.pt")

# Soglia confidenza — sotto questa soglia chiediamo all'LLM
CONFIDENCE_THRESHOLD = 0.50

# Predicati rari su cui attivare il fallback LLM
# (quelli con pochi esempi in training — < 200)
RARE_PREDICATES = {
    "strike", "change effect", "produced by", "compared to",
    "change expression", "change abundance", "part of"
}

# Path few-shot examples (generato da questo notebook)
FEWSHOT_PATH = Path("../data/few_shot_examples.json")

# Output
OUTPUT_PATH = Path("../predictions/inference_llm_fallback.json")
# ════════════════════════════════════════════

client = Groq(api_key=GROQ_API_KEY)
print("Groq client initialized")
print(f"Confidence threshold: {CONFIDENCE_THRESHOLD}")
print(f"Rare predicates: {RARE_PREDICATES}")


## Extract Few-Shot Examples from Gold Training Set

Run this once to generate `few_shot_examples.json`. Skip if already done.

In [ ]:
from collections import defaultdict
from pathlib import Path

GOLD_PATH = "../../../data/GutBrainIE_Full_Collection_2026/Annotations/Train/gold_quality/json_format/train_gold.json"

with open(GOLD_PATH) as f:
    gold_data = json.load(f)

pred_examples = defaultdict(list)

for pmid, doc in gold_data.items():
    title    = doc["metadata"]["title"]
    abstract = doc["metadata"]["abstract"]
    full_text = f"{title} {abstract}"
    abstract_offset = len(title) + 1

    entities = {}
    for e in doc["entities"]:
        key = (e["text_span"], e["label"])
        offset = abstract_offset if e["location"] == "abstract" else 0
        entities[key] = {
            "text_span":  e["text_span"],
            "label":      e["label"],
            "start_idx":  e["start_idx"] + offset,
            "end_idx":    e["end_idx"]   + offset,
        }

    for r in doc.get("mention_level_relations", []):
        pred = r["predicate"].strip()
        if len(pred_examples[pred]) >= 3:
            continue
        subj = entities.get((r["subject_text_span"], r["subject_label"]))
        obj  = entities.get((r["object_text_span"],  r["object_label"]))
        if not subj or not obj:
            continue
        left  = min(subj["start_idx"], obj["start_idx"])
        right = max(subj["end_idx"],   obj["end_idx"])
        win_start = max(0, left - 150)
        win_end   = min(len(full_text), right + 150)
        context = full_text[win_start:win_end].strip()
        pred_examples[pred].append({
            "context":       context,
            "subject":       r["subject_text_span"],
            "subject_label": r["subject_label"],
            "object":        r["object_text_span"],
            "object_label":  r["object_label"],
        })

# Salva
FEWSHOT_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(FEWSHOT_PATH, "w", encoding="utf-8") as f:
    json.dump(dict(pred_examples), f, ensure_ascii=False, indent=2)

print(f"Saved {len(pred_examples)} predicates to {FEWSHOT_PATH}")
for pred, exs in sorted(pred_examples.items(), key=lambda x: len(x[1])):
    print(f"  {pred}: {len(exs)} examples")


## Labels and Legal Pairs

In [ ]:
import re
from collections import Counter

LEGAL_RELATION_LABELS = {
    "administered","affect","change abundance","change effect","change expression","compared to",
    "impact","influence","interact","is a","is linked to","located in","part of","produced by",
    "strike","target","used by"
}
RELATION_LABELS = [
    "no relation","administered","affect","change abundance","change effect","change expression",
    "compared to","impact","influence","interact","is a","is linked to","located in","part of",
    "produced by","strike","target","used by"
]
label2id = {l: i for i, l in enumerate(RELATION_LABELS)}
id2label = {i: l for i, l in enumerate(RELATION_LABELS)}

LEGAL_RELATIONS = [
    ("DDF","affect","DDF"),("microbiome","is linked to","DDF"),("DDF","target","human"),
    ("drug","change effect","DDF"),("DDF","is a","DDF"),("microbiome","located in","human"),
    ("chemical","influence","DDF"),("dietary supplement","influence","DDF"),("DDF","target","animal"),
    ("chemical","impact","microbiome"),("anatomical location","located in","animal"),
    ("microbiome","located in","animal"),("chemical","located in","anatomical location"),
    ("bacteria","part of","microbiome"),("DDF","strike","anatomical location"),
    ("drug","administered","animal"),("bacteria","influence","DDF"),("drug","impact","microbiome"),
    ("DDF","change abundance","microbiome"),("microbiome","located in","anatomical location"),
    ("microbiome","used by","biomedical technique"),("chemical","produced by","microbiome"),
    ("dietary supplement","impact","microbiome"),("bacteria","located in","animal"),
    ("animal","used by","biomedical technique"),("chemical","impact","bacteria"),
    ("chemical","located in","animal"),("food","impact","bacteria"),
    ("microbiome","compared to","microbiome"),("human","used by","biomedical technique"),
    ("bacteria","change expression","gene"),("chemical","located in","human"),
    ("drug","interact","chemical"),("food","administered","human"),
    ("DDF","change abundance","bacteria"),("chemical","interact","chemical"),
    ("chemical","part of","chemical"),("dietary supplement","impact","bacteria"),
    ("DDF","interact","chemical"),("food","impact","microbiome"),("food","influence","DDF"),
    ("bacteria","located in","human"),("dietary supplement","administered","human"),
    ("bacteria","interact","chemical"),("drug","change expression","gene"),
    ("drug","impact","bacteria"),("drug","administered","human"),
    ("anatomical location","located in","human"),("dietary supplement","change expression","gene"),
    ("chemical","change expression","gene"),("bacteria","interact","bacteria"),
    ("drug","interact","drug"),("microbiome","change expression","gene"),
    ("bacteria","interact","drug"),("food","change expression","gene"),
]

def norm_ent(l):
    if not l: return ""
    return "DDF" if str(l).strip().lower() == "ddf" else str(l).strip()

def norm_span(s):
    return re.sub(r"\s+", " ", str(s).strip())

legal_pairs = {}
for s, p, o in LEGAL_RELATIONS:
    legal_pairs.setdefault((norm_ent(s), norm_ent(o)), set()).add(p)

print(f"Legal type pairs: {len(legal_pairs)}")


## Load Few-Shot Examples from Gold Training Set

In [ ]:
# Carica few-shot examples estratti dal gold training set
with open(FEWSHOT_PATH) as f:
    few_shot_examples = json.load(f)

print("Few-shot examples loaded:")
for pred, exs in few_shot_examples.items():
    print(f"  {pred}: {len(exs)} examples")


## Prompt Builder

In [ ]:
def build_prompt(context, subject, subject_label, obj, obj_label,
                  few_shot_examples, legal_predicates):
    """
    Builds a few-shot prompt for a single entity pair.
    Only includes examples for predicates legal for this entity type pair.
    """
    lines = []
    lines.append("You are an expert biomedical relation extraction system.")
    lines.append("Given a biomedical text and two entities, classify their relation.")
    lines.append(f"The subject entity type is: {subject_label}")
    lines.append(f"The object entity type is: {obj_label}")
    lines.append(f"Legal relations for this entity pair: {', '.join(sorted(legal_predicates))}")
    lines.append("")
    lines.append("Output ONLY the relation label, nothing else.")
    lines.append("If no relation holds, output: no relation")
    lines.append("")
    lines.append("--- EXAMPLES ---")

    # Aggiungi few-shot per i predicati legali di questa coppia
    n_shown = 0
    for pred in sorted(legal_predicates):
        if pred not in few_shot_examples:
            continue
        for ex in few_shot_examples[pred][:2]:  # max 2 per predicato
            lines.append(f"Text: {ex['context'][:300]}")
            lines.append(f"Subject: {ex['subject']} [{ex['subject_label']}]")
            lines.append(f"Object: {ex['object']} [{ex['object_label']}]")
            lines.append(f"Relation: {pred}")
            lines.append("")
            n_shown += 1
            if n_shown >= 6:  # max 6 esempi totali per tenere il prompt corto
                break
        if n_shown >= 6:
            break

    # Aggiungi un esempio "no relation" generico
    lines.append("Text: The gut microbiota was analyzed using 16S rRNA sequencing in healthy subjects.")
    lines.append("Subject: gut microbiota [microbiome]")
    lines.append("Object: Parkinson disease [DDF]")
    lines.append("Relation: no relation")
    lines.append("")

    lines.append("--- CLASSIFY ---")
    lines.append(f"Text: {context[:400]}")
    lines.append(f"Subject: {subject} [{subject_label}]")
    lines.append(f"Object: {obj} [{obj_label}]")
    lines.append("Relation:")

    return "\n".join(lines)


def call_groq(prompt, model="llama-3.1-70b-versatile", max_retries=3):
    """Call Groq API with retry on rate limit."""
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": prompt}],
                max_tokens=20,
                temperature=0.0,
            )
            answer = response.choices[0].message.content.strip().lower()
            return answer
        except Exception as e:
            if "rate" in str(e).lower() and attempt < max_retries - 1:
                time.sleep(2 ** attempt)
            else:
                return None
    return None


def parse_llm_answer(answer, legal_predicates):
    """Parse LLM output to a valid predicate or no relation."""
    if answer is None:
        return "no relation"
    answer = answer.strip().lower().rstrip(".")
    # Match esatto
    if answer in legal_predicates:
        return answer
    if answer == "no relation":
        return "no relation"
    # Match parziale
    for pred in legal_predicates:
        if pred in answer:
            return pred
    return "no relation"


print("Prompt builder and Groq caller defined")


## Load BERT Logits Cache and Identify Low-Confidence Pairs

In [ ]:
def softmax_np(x):
    import numpy as np
    x = x - x.max()
    e = __import__("math").e
    ex = [e**v for v in x]
    s = sum(ex)
    return [v/s for v in ex]

# Carica logits cache BERT
print(f"Loading BERT logits cache from: {BERT_LOGITS_CACHE}")
bert_cache = torch.load(BERT_LOGITS_CACHE, map_location="cpu")
print(f"Documents in cache: {len(bert_cache)}")
print(f"Total pairs: {sum(len(v) for v in bert_cache.values())}")

# Identifica coppie low-confidence su predicati rari
import numpy as np

low_conf_pairs = []  # (pmid, row, bert_pred, bert_conf)
bert_decisions  = {}  # pmid -> {k: (pred, conf)}

for pmid, rows in bert_cache.items():
    bert_decisions[pmid] = {}
    for row in rows:
        s_lab = row["subject_label"]
        o_lab = row["object_label"]
        allowed = sorted(legal_pairs.get((s_lab, o_lab), []))
        if not allowed:
            continue

        logits = row["logits"].numpy()
        allowed_ids = [label2id["no relation"]] + [label2id[p] for p in allowed]
        probs_sub = np.array(logits[allowed_ids])
        probs_sub = np.exp(probs_sub - probs_sub.max())
        probs_sub /= probs_sub.sum()

        p_no = float(probs_sub[0])
        rel_probs = probs_sub[1:]
        best_idx = int(np.argmax(rel_probs))
        bert_pred = allowed[best_idx]
        bert_conf = float(rel_probs[best_idx])

        k = row["k"]
        # Tieni il best per chiave
        prev = bert_decisions[pmid].get(k)
        if prev is None or bert_conf > prev[1]:
            bert_decisions[pmid][k] = (bert_pred, bert_conf)

        # Segna come low-confidence se sotto soglia E predicato raro
        if bert_conf < CONFIDENCE_THRESHOLD and bert_pred in RARE_PREDICATES:
            low_conf_pairs.append((pmid, row, bert_pred, bert_conf))

print(f"\nLow-confidence pairs on rare predicates: {len(low_conf_pairs)}")
pred_counts = Counter(x[2] for x in low_conf_pairs)
print("By predicate:")
for pred, cnt in pred_counts.most_common():
    print(f"  {pred}: {cnt}")


## Run LLM Fallback on Low-Confidence Pairs

In [ ]:
# Carica dev data per il contesto testuale
import os

DEV_PATH = "../../../data/GutBrainIE_Full_Collection_2026/Annotations/Dev/json_format/dev.json"
with open(DEV_PATH) as f:
    dev_data = json.load(f)

def get_context(pmid, row, window_chars=300):
    """Recupera il testo di contesto per una coppia dal dev set."""
    art = dev_data.get(str(pmid)) or dev_data.get(pmid)
    if not art:
        return ""
    title = art["metadata"]["title"]
    abstract = art["metadata"]["abstract"]
    full_text = f"{title} {abstract}"
    return full_text

print(f"Dev data loaded: {len(dev_data)} documents")
print(f"\nRunning LLM fallback on {len(low_conf_pairs)} pairs...")
print("This will use Groq API — estimated time: ~1-2 min")
print()

llm_overrides = {}  # (pmid, k) -> llm_pred

api_calls = 0
skipped   = 0

for pmid, row, bert_pred, bert_conf in low_conf_pairs:
    s_lab = row["subject_label"]
    o_lab = row["object_label"]
    k     = row["k"]
    s_text, s_lab_k, o_text, o_lab_k = k

    legal_preds = legal_pairs.get((s_lab, o_lab), set())
    if not legal_preds:
        skipped += 1
        continue

    context = get_context(pmid, row)
    if not context:
        skipped += 1
        continue

    prompt = build_prompt(
        context=context,
        subject=s_text, subject_label=s_lab,
        obj=o_text, obj_label=o_lab,
        few_shot_examples=few_shot_examples,
        legal_predicates=legal_preds,
    )

    llm_raw = call_groq(prompt)
    llm_pred = parse_llm_answer(llm_raw, legal_preds)
    api_calls += 1

    # Salva override solo se LLM predice una relazione (non no relation)
    # e diversa da BERT
    if llm_pred != "no relation":
        llm_overrides[(str(pmid), k)] = llm_pred

    if api_calls % 20 == 0:
        print(f"  {api_calls}/{len(low_conf_pairs)} calls done...")
    
    time.sleep(0.1)  # gentile con le rate limits

print(f"\nDone. API calls: {api_calls} | Skipped: {skipped}")
print(f"LLM overrides applied: {len(llm_overrides)}")
override_preds = Counter(v for v in llm_overrides.values())
print("Override distribution:", dict(override_preds))


## Merge BERT + LLM and Evaluate

In [ ]:
def build_gold_maps(dev_data):
    LEGAL_RELATION_LABELS = set(RELATION_LABELS) - {"no relation"}
    gold = {}
    for pmid, art in dev_data.items():
        s = set()
        for r in art.get("mention_level_relations", []):
            pred = r["predicate"].strip()
            if pred not in LEGAL_RELATION_LABELS:
                continue
            s.add((
                norm_span(r["subject_text_span"]), norm_ent(r["subject_label"]),
                pred,
                norm_span(r["object_text_span"]),  norm_ent(r["object_label"]),
            ))
        gold[str(pmid)] = s
    return gold

def micro_scores(gold, pred):
    tp=fp=fn=0
    for pmid, g in gold.items():
        p = pred.get(pmid, set())
        tp+=len(g&p); fp+=len(p-g); fn+=len(g-p)
    P = tp/(tp+fp) if (tp+fp) else 0.0
    R = tp/(tp+fn) if (tp+fn) else 0.0
    F1 = 2*P*R/(P+R) if (P+R) else 0.0
    return {"P":round(P,4),"R":round(R,4),"F1":round(F1,4),"TP":tp,"FP":fp,"FN":fn}

def macro_scores(gold, pred):
    preds = sorted({t[2] for s in gold.values() for t in s})
    vals = []
    for pr in preds:
        tp=fp=fn=0
        for pmid, g in gold.items():
            gp={t for t in g if t[2]==pr}
            pp={t for t in pred.get(pmid,set()) if t[2]==pr}
            tp+=len(gp&pp); fp+=len(pp-gp); fn+=len(gp-pp)
        P=tp/(tp+fp) if (tp+fp) else 0.0
        R=tp/(tp+fn) if (tp+fn) else 0.0
        vals.append((P,R,2*P*R/(P+R) if (P+R) else 0.0))
    import numpy as np
    return {
        "macro_P": round(float(np.mean([x[0] for x in vals])),4),
        "macro_R": round(float(np.mean([x[1] for x in vals])),4),
        "macro_F1": round(float(np.mean([x[2] for x in vals])),4),
    }

gold_by_doc = build_gold_maps(dev_data)

# ── BERT-only predictions ──
bert_pred_by_doc = {}
for pmid, decisions in bert_decisions.items():
    s = set()
    for k, (pred, conf) in decisions.items():
        if pred != "no relation" and pred in LEGAL_RELATION_LABELS:
            st, sl, ot, ol = k
            s.add((norm_span(st), norm_ent(sl), pred, norm_span(ot), norm_ent(ol)))
    bert_pred_by_doc[str(pmid)] = s

# ── BERT + LLM predictions ──
merged_pred_by_doc = {pmid: set(preds) for pmid, preds in bert_pred_by_doc.items()}

for (pmid, k), llm_pred in llm_overrides.items():
    st, sl, ot, ol = k
    triple = (norm_span(st), norm_ent(sl), llm_pred, norm_span(ot), norm_ent(ol))
    merged_pred_by_doc.setdefault(pmid, set()).add(triple)

# ── Evaluation ──
bert_mi = micro_scores(gold_by_doc, bert_pred_by_doc)
bert_ma = macro_scores(gold_by_doc, bert_pred_by_doc)

merged_mi = micro_scores(gold_by_doc, merged_pred_by_doc)
merged_ma = macro_scores(gold_by_doc, merged_pred_by_doc)

print("=== BERT only ===")
print(f"  Macro  P={bert_ma['macro_P']:.4f}  R={bert_ma['macro_R']:.4f}  F1={bert_ma['macro_F1']:.4f}")
print(f"  Micro  P={bert_mi['P']:.4f}  R={bert_mi['R']:.4f}  F1={bert_mi['F1']:.4f}")

print("\n=== BERT + LLM fallback ===")
print(f"  Macro  P={merged_ma['macro_P']:.4f}  R={merged_ma['macro_R']:.4f}  F1={merged_ma['macro_F1']:.4f}")
print(f"  Micro  P={merged_mi['P']:.4f}  R={merged_mi['R']:.4f}  F1={merged_mi['F1']:.4f}")

delta_macro = merged_ma['macro_F1'] - bert_ma['macro_F1']
delta_micro = merged_mi['F1'] - bert_mi['F1']
print(f"\n  Delta Macro F1: {delta_macro:+.4f}")
print(f"  Delta Micro F1: {delta_micro:+.4f}")


## Save Predictions

In [ ]:
# Build submission format
predictions_out = {}
for pmid, pred_set in merged_pred_by_doc.items():
    relations = []
    for (st, sl, pred, ot, ol) in sorted(pred_set):
        relations.append({
            "subject_text_span": st,
            "subject_label": sl,
            "predicate": pred,
            "object_text_span": ot,
            "object_label": ol,
        })
    predictions_out[pmid] = {"mention_level_relations": relations}

os.makedirs(OUTPUT_PATH.parent, exist_ok=True)
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(predictions_out, f, ensure_ascii=False, indent=2)

print(f"Saved: {OUTPUT_PATH}")
print(f"Total documents: {len(predictions_out)}")
print(f"Total relations: {sum(len(v['mention_level_relations']) for v in predictions_out.values())}")
